In [49]:
from pyspark.sql import SparkSession

In [50]:
spark = SparkSession.builder.appName("app").master("local[*]").getOrCreate()

In [51]:
df = spark.read.csv("data/raw/flight_data.csv", header=True, inferSchema=True)

In [52]:
df.columns

['legId',
 'searchDate',
 'flightDate',
 'startingAirport',
 'destinationAirport',
 'fareBasisCode',
 'travelDuration',
 'elapsedDays',
 'isBasicEconomy',
 'isRefundable',
 'isNonStop',
 'baseFare',
 'totalFare',
 'seatsRemaining',
 'totalTravelDistance',
 'segmentsDepartureTimeEpochSeconds',
 'segmentsDepartureTimeRaw',
 'segmentsArrivalTimeEpochSeconds',
 'segmentsArrivalTimeRaw',
 'segmentsArrivalAirportCode',
 'segmentsDepartureAirportCode',
 'segmentsAirlineName',
 'segmentsAirlineCode',
 'segmentsEquipmentDescription',
 'segmentsDurationInSeconds',
 'segmentsDistance',
 'segmentsCabinCode']

In [53]:
df.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTimeEpochSeconds: string (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsArrivalTimeEpochSeconds: string (nullable = true)
 |-- segmentsArrivalTimeRaw: string (nullable = true)
 |-- segmentsArrivalAirportCode: string (nullable = true)
 |-- segmentsDepartureAirportCode: s

In [54]:
df.select("segmentsCabinCode").show()

+-----------------+
|segmentsCabinCode|
+-----------------+
|            coach|
|            coach|
|            coach|
|            coach|
|            coach|
|            coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
|            coach|
|     coach||coach|
|     coach||coach|
|     coach||coach|
+-----------------+
only showing top 20 rows


## Preprocessing

In [55]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, BooleanType, StringType, ArrayType

### 1. Drop rows with null / blank values

In [56]:
before = df.count()

# Drop rows with any true null
df1 = df.dropna()

# Drop rows where any scalar string column is blank
scalar_str_cols = [f.name for f in df1.schema.fields if isinstance(f.dataType, StringType)]
for c in scalar_str_cols:
    df1 = df1.filter(F.col(c) != "")

# Drop rows where any string array is empty or contains a null/blank element
# (no array columns at this stage — loop is a no-op, kept for consistency)
array_str_cols = [
    f.name for f in df1.schema.fields
    if isinstance(f.dataType, ArrayType) and isinstance(f.dataType.elementType, StringType)
]
for c in array_str_cols:
    df1 = df1.filter(
        (F.size(F.col(c)) > 0) &
        (~F.exists(F.col(c), lambda x: x.isNull() | (F.trim(x) == "")))
    )

after = df1.count()
print(f"Dropped {before - after:,} rows  ({before:,} → {after:,})")

Dropped 7,384,463 rows  (82,138,753 → 74,754,290)


### 2. Date fields → DateType  
### + Boolean columns → BooleanType (additional)

In [57]:
df2 = (
    df1
    # Date fields
    .withColumn("searchDate", F.to_date("searchDate", "yyyy-MM-dd"))
    .withColumn("flightDate",  F.to_date("flightDate",  "yyyy-MM-dd"))
    # Boolean columns — Spark reads "True"/"False" strings when inferSchema misses them
    .withColumn("isBasicEconomy", F.col("isBasicEconomy").cast(BooleanType()))
    .withColumn("isRefundable",   F.col("isRefundable").cast(BooleanType()))
    .withColumn("isNonStop",      F.col("isNonStop").cast(BooleanType()))
)

df2.select("searchDate", "flightDate", "isBasicEconomy", "isRefundable", "isNonStop").printSchema()

root
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)



### 3. `travelDuration` → integer minutes

In [58]:
# ISO 8601 duration format: PT2H29M / PT30M / PT2H
# regexp_extract returns "" (empty string) when the group doesn't match — handled with when/otherwise
_hours = F.when(
    F.regexp_extract("travelDuration", r"(\d+)H", 1) != "",
    F.regexp_extract("travelDuration", r"(\d+)H", 1).cast("int")
).otherwise(F.lit(0))

_mins = F.when(
    F.regexp_extract("travelDuration", r"(\d+)M", 1) != "",
    F.regexp_extract("travelDuration", r"(\d+)M", 1).cast("int")
).otherwise(F.lit(0))

df3 = df2.withColumn("travelDuration", (_hours * 60 + _mins).cast(IntegerType()))

df3.select("travelDuration").show(5)

+--------------+
|travelDuration|
+--------------+
|           149|
|           150|
|           150|
|           152|
|           154|
+--------------+
only showing top 5 rows


### 4. Split `segments*` columns on `||` → ArrayType  
### + Cast numeric array elements (additional)

In [59]:
segment_cols = [c for c in df3.columns if c.startswith("segments")]

# Split every segment column on || → array<string>
df4 = df3
for col_name in segment_cols:
    df4 = df4.withColumn(col_name, F.split(F.col(col_name), r"\|\|"))

# Helper: replace the literal string "None" with null before casting
# (CSV may contain Python's str(None) which ANSI-mode Spark refuses to cast)
def safe_cast(col_expr, dtype):
    return F.transform(
        col_expr,
        lambda x: F.when((x == "None") | x.isNull(), F.lit(None).cast(dtype)).otherwise(x.cast(dtype))
    )

df4 = (
    df4
    .withColumn("segmentsDurationInSeconds",
                safe_cast(F.col("segmentsDurationInSeconds"), IntegerType()))
    .withColumn("segmentsDistance",
                safe_cast(F.col("segmentsDistance"), IntegerType()))
    .withColumn("segmentsDepartureTimeEpochSeconds",
                safe_cast(F.col("segmentsDepartureTimeEpochSeconds"), LongType()))
    .withColumn("segmentsArrivalTimeEpochSeconds",
                safe_cast(F.col("segmentsArrivalTimeEpochSeconds"), LongType()))
)

df4.select(*segment_cols).printSchema()

root
 |-- segmentsDepartureTimeEpochSeconds: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- segmentsDepartureTimeRaw: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsArrivalTimeEpochSeconds: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- segmentsArrivalTimeRaw: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsArrivalAirportCode: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsDepartureAirportCode: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsAirlineName: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsAirlineCode: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsEquipmentDescription: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsDurationInSeconds: array (nullable = t

### 5. `segmentsCabinCode` — replace cabin code labels

In [60]:
df5 = df4.withColumn(
    "segmentsCabinCode",
    F.transform(
        F.col("segmentsCabinCode"),
        lambda x: F.when(x == "coach", "economy")
                   .when(x == "premium coach", "premium economy")
                   .otherwise(x)
    )
)

df5.select("segmentsCabinCode").show(10, truncate=False)

+------------------+
|segmentsCabinCode |
+------------------+
|[economy]         |
|[economy]         |
|[economy]         |
|[economy]         |
|[economy]         |
|[economy, economy]|
|[economy, economy]|
|[economy, economy]|
|[economy, economy]|
|[economy, economy]|
+------------------+
only showing top 10 rows


### 6. Drop epoch seconds columns

In [61]:
df6 = df5.drop("segmentsDepartureTimeEpochSeconds", "segmentsArrivalTimeEpochSeconds")

### 7. Add derived columns

In [62]:
# numDaysToFlight: days between searchDate and flightDate
df7 = df6.withColumn("numDaysToFlight", F.datediff(F.col("flightDate"), F.col("searchDate")))

# numLayovers: number of connecting segments minus the origin leg
# size(segmentsDepartureAirportCode) - 1  →  0 for direct, 1 for one stop, etc.
df7 = df7.withColumn(
    "numLayovers",
    F.greatest(F.size(F.col("segmentsDepartureAirportCode")) - 1, F.lit(0))
)

# layoverDurationMinutes: sum of wait times between consecutive legs
# For each connection i: departure[i] - arrival[i-1] (in seconds), then convert to minutes
_dep_epoch = F.transform(
    F.col("segmentsDepartureTimeRaw"),
    lambda x: F.unix_timestamp(x, "yyyy-MM-dd'T'HH:mm:ss.SSSXXX")
)
_arr_epoch = F.transform(
    F.col("segmentsArrivalTimeRaw"),
    lambda x: F.unix_timestamp(x, "yyyy-MM-dd'T'HH:mm:ss.SSSXXX")
)
_n = F.size(F.col("segmentsDepartureTimeRaw"))

# zip departure[1..n-1] with arrival[0..n-2] and take the difference
_layover_seconds = F.zip_with(
    F.slice(_dep_epoch, 2, _n - 1),
    F.slice(_arr_epoch, 1, _n - 1),
    lambda dep, arr: dep - arr
)

df7 = df7.withColumn(
    "layoverDurationMinutes",
    (F.aggregate(
        _layover_seconds,
        F.lit(0).cast("long"),
        lambda acc, x: acc + F.coalesce(x, F.lit(0).cast("long"))
    ) / 60).cast(IntegerType())
)

df7.select("numDaysToFlight", "numLayovers", "layoverDurationMinutes").show(10)

+---------------+-----------+----------------------+
|numDaysToFlight|numLayovers|layoverDurationMinutes|
+---------------+-----------+----------------------+
|              1|          0|                     0|
|              1|          0|                     0|
|              1|          0|                     0|
|              1|          0|                     0|
|              1|          0|                     0|
|              1|          1|                    37|
|              1|          1|                    90|
|              1|          1|                   126|
|              1|          1|                   179|
|              1|          1|                    79|
+---------------+-----------+----------------------+
only showing top 10 rows


In [63]:
df7.select("numDaysToFlight", "numLayovers", "layoverDurationMinutes").show(5)

+---------------+-----------+----------------------+
|numDaysToFlight|numLayovers|layoverDurationMinutes|
+---------------+-----------+----------------------+
|              1|          0|                     0|
|              1|          0|                     0|
|              1|          0|                     0|
|              1|          0|                     0|
|              1|          0|                     0|
+---------------+-----------+----------------------+
only showing top 5 rows


### 8. Timestamp arrays → TimestampType, duration → minutes, column renames

In [64]:
_ts_fmt = "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"

df8 = (
    df7
    # Parse timestamp string arrays → array<timestamp>, truncated to second precision
    .withColumn(
        "segmentsDepartureTimeRaw",
        F.transform(F.col("segmentsDepartureTimeRaw"),
                    lambda x: F.date_trunc("second", F.to_timestamp(x, _ts_fmt)))
    )
    .withColumn(
        "segmentsArrivalTimeRaw",
        F.transform(F.col("segmentsArrivalTimeRaw"),
                    lambda x: F.date_trunc("second", F.to_timestamp(x, _ts_fmt)))
    )
    # segmentsDurationInSeconds → segmentsDurationInMinutes (rounded int)
    .withColumn(
        "segmentsDurationInMinutes",
        F.transform(F.col("segmentsDurationInSeconds"), lambda x: F.round(x / 60).cast(IntegerType()))
    )
    .drop("segmentsDurationInSeconds")
    # Remove 'Raw' suffix
    .withColumnRenamed("segmentsDepartureTimeRaw", "segmentsDepartureTime")
    .withColumnRenamed("segmentsArrivalTimeRaw",   "segmentsArrivalTime")
    # Rename equipment description
    .withColumnRenamed("segmentsEquipmentDescription", "segmentsAircraft")
)

df8.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: integer (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTime: array (nullable = true)
 |    |-- element: timestamp (containsNull = true)
 |-- segmentsArrivalTime: array (nullable = true)
 |    |-- element: timestamp (containsNull = true)
 |-- segmentsArrivalAirportCode: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- segmentsDepartur

### 9. Reorder columns

In [68]:
ordered_cols = [
    # Route identifiers
    "legId",
    "searchDate", "flightDate",
    "startingAirport", "destinationAirport",
    # Derived travel info
    "numDaysToFlight", "numLayovers", "layoverDurationMinutes",
    # Availability
    "seatsRemaining",
    # Flight characteristics
    "travelDuration", "totalTravelDistance", "elapsedDays",
    "isBasicEconomy", "isRefundable", "isNonStop",
    # Flight Fare
    "baseFare", "totalFare", "fareBasisCode",
    # Segment arrays
    "segmentsDepartureTime", "segmentsArrivalTime",
    "segmentsArrivalAirportCode", "segmentsDepartureAirportCode",
    "segmentsAirlineName", "segmentsAirlineCode",
    "segmentsAircraft", "segmentsDurationInMinutes",
    "segmentsDistance", "segmentsCabinCode",
]

df9 = df8.select(ordered_cols)
df9.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- numDaysToFlight: integer (nullable = true)
 |-- numLayovers: integer (nullable = false)
 |-- layoverDurationMinutes: integer (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- travelDuration: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- segmentsDepartureTime: array (nullable = true)
 |    |-- element: timestamp (containsNull = true)
 |-- segmentsArrivalTime: array (nullable = true)
 |    |-- element: timestamp (

### 10. Save as Parquet

In [69]:
output_path = "data/flights_clean_complete.parquet"

df9.write.mode("overwrite").parquet(output_path)

print(f"Saved {df9.count():,} rows → {output_path}")

26/03/11 12:17:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/11 12:17:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/11 12:17:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/11 12:17:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/11 12:17:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/11 12:17:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/11 12:17:20 WARN MemoryManager: Total allocation exceeds 95.

Saved 74,754,290 rows → data/flights_clean_complete.parquet
